# Reworked Pipeline — Uçtan Uca Workflow

Bu notebook, **yeni (reworked) hiyerarşik GNN policy** stack'inin tam akışını baştan sona anlatır:

1. Ham env gözlemi
2. `obs_builder` → `Observation`
3. `reworked_pipeline` → node/edge/global feature tensörleri
4. `reworked_model` → GNN encoder + stratejik karar head'leri
5. `reworked_allocator` → lojistik (gemi yönlendirme)
6. `env.step` → ödül (potential-based shaping)
7. `reworked_rollout` → buffer, GAE, PPO
8. `reworked_train` → eğitim döngüsü

Legacy stack (`rl/train.py`, otoregresif launch head'leri, arrival-credit ödül zoo'su) **bilerek kullanılmıyor**. Bu defter yalnızca `reworked_*.py` modüllerini kapsar.

Detaylı metrik formülleri için: `docs/reworked_features_reference.md`

## Mimari özeti

```
env.observation (ham dict)
      │
      ▼
obs_builder.build_observation()  ──► Observation
      │   planet_feats (34-D, normalised)  planet_mask
      │   global_feats (14-D)              edge_bias (P×P×6)
      │   fleet routing arrays             planet_ids
      ▼
reworked_pipeline
      │   board_state_from_obs()  → owner, ships, production, distance_matrix …
      │   build_reworked_features() → node (15-D), edge (4-D), global (14-D)
      │   pack_reworked_batch()     → torch tensors (B, P, …)
      ▼
ReworkedHierarchicalPolicy
      │   ReworkedPlanetGraphEncoder (Dense GINE, n_layers)
      │     → node_tokens, edge_tokens, graph_summary
      │   Stratejik head'ler (RL kararları):
      │     attack_head / defend_head   → multi-label Bernoulli (hedef seçimi)
      │     priority_head + PL          → Plackett-Luce sıralama (kıtlıkta öncelik)
      │     commit_head                 → Beta(α,β) per-planet (taahhüt oranı)
      │     posture_head                → global potential_weight + horizon
      │     value_head                  → critic V(s)
      ▼
policy.act()  → {attack_sel, defend_sel, order, commit_frac, posture_u, log_prob, value}
      │
      ▼
reworked_allocator.allocate_launches()  (MCF veya greedy)
      │   RL'nin seçtiği hedefler + öncelik + commit → concrete Launch list
      ▼
navigation.compute_launch()  → [[planet_id, angle, ships], …]
      │
      ▼
env.step()  → next observation
      │
      ▼
economic_potential Φ(s)  →  r = γ·Φ(s') − Φ(s)  (+ terminal ±1)
      │
      ▼
ReworkedBuffer  →  compute_gae()  →  ppo_update()
```

**Temel tasarım kararı:** RL *ne* saldırılacak/savunulacak, *öncelik* ve *risk* kararlarını verir; *hangi kaynaktan kaç gemi* gideceğini scripted allocator çözer.

## 0) Kurulum

Notebook `docs/` altından çalıştırılırsa proje köküne (`Orbit_Wars/`) geçer.

In [ ]:
import os, sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "reworked_model.py").exists():
    ROOT = ROOT.parent  # docs/ → Orbit_Wars/
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import torch

torch.manual_seed(0)
np.random.seed(0)
DEVICE = torch.device("cpu")
print("ROOT:", ROOT)
print("torch:", torch.__version__, "device:", DEVICE)

## 1) Ham env gözlemi ve oyun başlatma

`kaggle_environments.make("orbit_wars")` ile 2 oyunculu self-play ortamı açılır. Her oyuncunun `observation` dict'i gezegen listesi, filo listesi, açısal hız vb. içerir.

In [ ]:
from kaggle_environments import make

env = make("orbit_wars", debug=False)
env.configuration["randomSeed"] = 42
env.reset(num_agents=2)
state = env.step([[], []])  # ilk tur boş aksiyon

PLAYER = 0
env_obs = state[PLAYER]["observation"]
print("planets:", len(env_obs["planets"]))
print("fleets:", len(env_obs.get("fleets", [])))
print("örnek gezegen satırı [id, owner, x, y, radius, ships, production]:")
print(env_obs["planets"][0])

## 2) Observation inşası — `obs_builder.build_observation`

`obs_builder` ham dict'i sabit boyutlu NumPy dizilerine çevirir:

| Çıktı | Boyut | Açıklama |
|-------|-------|----------|
| `planet_feats` | `(P_MAX, 34)` | Normalise edilmiş gezegen özellikleri (ownership one-hot, ships/100, prod/5, …) |
| `planet_mask` | `(P_MAX,)` | Gerçek gezegen slotları |
| `global_feats` | `(14,)` | Tur, kuvvet dengesi, üretim, incoming tehdit, … |
| `edge_bias` | `(P_MAX, P_MAX, 6)` | Çiftler arası mesafe, crosses_sun, … |
| `fleet_*` | çeşitli | Havadaki filoların slot pointer'ları |

Reworked pipeline bu `Observation` nesnesini **giriş noktası** olarak kullanır.

In [ ]:
from obs_builder import build_observation, GLOBAL_FEATURE_DIM, PLANET_FEATURE_DIM

TURN, MAX_TURNS = 0, 300
P_MAX, F_MAX = 44, 64
av = env_obs.get("angular_velocity", 0.0) or 0.0

obs = build_observation(
    env_obs, player=PLAYER, angular_velocity=av,
    p_max=P_MAX, f_max=F_MAX, turn=TURN, max_turns=MAX_TURNS,
)

n_real = int(obs.planet_mask.sum())
print(f"planet_feats: {obs.planet_feats.shape}  (gerçek gezegen: {n_real})")
print(f"global_feats: {obs.global_feats.shape}")
print(f"edge_bias:    {obs.edge_bias.shape}")
print(f"global (my_ships, enemy_ships, incoming_other):",
      obs.global_feats[5], obs.global_feats[6], obs.global_feats[10])

## 3) BoardState — de-normalise fiziksel durum

`reworked_pipeline.board_state_from_obs` normalise `Observation`'ı metrik fonksiyonlarının beklediği fiziksel birime çevirir:

- `ships × 100`, `production × 5`, `radius × 5`
- `effective_owner_from_obs`: perspektif-göreli ownership one-hot → mutlak owner id
- `edge_bias[..., 0]` → normalise mesafe matrisi (× `BOARD_SIZE` → ham birim)
- `edge_bias[..., 5]` → `crosses_sun` maskesi

In [ ]:
from reworked_pipeline import board_state_from_obs, effective_owner_from_obs

state_board = board_state_from_obs(obs, PLAYER, remaining_turns=MAX_TURNS - TURN)
print("BoardState:")
print("  n:", state_board.n)
print("  owner:", state_board.owner)
print("  ships:", np.round(state_board.ships, 1))
print("  production:", np.round(state_board.production, 2))
print("  distance[0,:3]:", np.round(state_board.distance_matrix[0, :3], 3))

## 4) Node feature'ları (15-D)

Her gezegen için **10 standart + 5 türetilmiş** kanal:

**Standart (10)** — `reworked_features.standard_node_features`:
ownership one-hot (5), ships/100, production/5, radius/5, is_rotating, is_comet

**Türetilmiş (5)** — `reworked_features`:

| # | Metrik | Kısa anlam |
|---|--------|------------|
| 1 | `discounted_production_potential` | Kalan ufukta üreteceği gemi (iskonto) |
| 2 | `closeness_centrality` | Geometrik merkezilik (owner-bağımsız) |
| 3 | `potential_threat` | Yakındaki düşman garnizon baskısı (hız-duyarlı) |
| 4 | `potential_support` | Yakındaki dost garnizon desteği |
| 5 | `friendly_compactness` | Dost toprak içinde ne kadar "derin" |

Formüller: `docs/reworked_features_reference.md`

In [ ]:
from reworked_pipeline import build_reworked_node_features, build_reworked_features
from reworked_model import REWORKED_NODE_FEATURE_DIM

node_feats = build_reworked_node_features(state_board)
print(f"node_feats shape: {node_feats.shape}  (beklenen: ({state_board.n}, {REWORKED_NODE_FEATURE_DIM}))")

labels = [
    "dpp", "closeness", "pot_threat", "pot_support", "compactness",
]
derived = node_feats[:, 10:]  # son 5 kanal
for j, name in enumerate(labels):
    print(f"  {name:14s} min={derived[:,j].min():.3f}  max={derived[:,j].max():.3f}  mean={derived[:,j].mean():.3f}")

## 5) Edge feature'ları (4-D)

Her yönlü gezegen çifti `(h → T)` için:

`[distance, crosses_sun, friendly_flux, enemy_flux]`

- **distance / crosses_sun**: geometri (`edge_bias`'ten)
- **friendly_flux / enemy_flux**: o yönde uçan dost/düşman filoların ağırlıklı akısı (`exp_decay` ile yakın varışlar daha ağır)

GNN mesaj geçişinde edge token olarak kullanılır (GINE conditioning).

In [ ]:
from reworked_pipeline import build_reworked_edge_features
from reworked_model import REWORKED_EDGE_FEATURE_DIM

edge_feats = build_reworked_edge_features(obs, state_board)
print(f"edge_feats shape: {edge_feats.shape}  (beklenen: ({state_board.n},{state_board.n},{REWORKED_EDGE_FEATURE_DIM}))")

# Gerçek gezegenler arası ortalama flux
n = state_board.n
sub = edge_feats[:n, :n]
for ch, name in enumerate(["distance", "crosses_sun", "friendly_flux", "enemy_flux"]):
    print(f"  {name:14s} mean={sub[..., ch].mean():.4f}")

## 6) Batch paketleme — `pack_reworked_batch`

Eğitimde sabit `P_MAX=44` padding ile batch'lenir. Çıktı PyTorch tensörleri:

- `node_feats` `(B, P, 15)`
- `edge_feats` `(B, P, P, 4)`
- `global_feats` `(B, 14)`
- `planet_mask` `(B, P)` bool

In [ ]:
from reworked_pipeline import pack_reworked_batch

batch = pack_reworked_batch([(obs, PLAYER)], device=DEVICE, p_max=P_MAX)
for k, v in batch.items():
    if hasattr(v, "shape"):
        print(f"  {k:16s} {tuple(v.shape)}  {v.dtype}")
    elif k == "states":
        print(f"  states           list[{len(v)}] BoardState")

## 7) Neural network — encoder + hiyerarşik policy

`ReworkedPlanetGraphEncoder`:
1. Input stem: node / edge / global → `d_model`
2. `n_layers` × `DenseGINELayer` (tam bağlı, GINE-equivalent message passing)
3. Masked mean/max pool + global embed → `graph_summary` `(B, d_model)`

`ReworkedHierarchicalPolicy` head'leri `[node_token, broadcast graph_summary]` alır — yani **her gezegen kararı global bağlamı görür** (kuvvet dengesi, gemi kıtlığı vb.).

Tur başına tek joint aksiyon:
- **attack_sel / defend_sel**: bağımsız Bernoulli (multi-label hedef seçimi)
- **order**: Plackett-Luce sıralama (seçilen hedefler arasında öncelik)
- **commit_frac**: kaynak başına Beta dağılımı (allocator ne kadar garnizon kullanabilir)
- **posture**: global `potential_weight` + `horizon` (savunma proaktifliği)

In [ ]:
from reworked_model import ReworkedHierarchicalPolicy
from reworked_rollout import _target_masks

policy = ReworkedHierarchicalPolicy(d_model=64, n_layers=2).to(DEVICE)
n_params = sum(p.numel() for p in policy.parameters())
print(f"Policy params: {n_params:,}")

st0 = batch["states"][0]
attack_np, defend_np = _target_masks(st0, P_MAX, PLAYER)
attack_mask = torch.from_numpy(attack_np)[None].to(DEVICE)
defend_mask = torch.from_numpy(defend_np)[None].to(DEVICE)

out = policy(
    node_feats=batch["node_feats"],
    edge_feats=batch["edge_feats"],
    global_feats=batch["global_feats"],
    planet_mask=batch["planet_mask"],
    attack_mask=attack_mask,
    defend_mask=defend_mask,
)
print("forward keys:", sorted(out.keys()))
print("attack_logits:", out["attack_logits"].shape)
print("value:", float(out["value"][0]))

In [ ]:
# Örnekleme: tam hiyerarşik aksiyon + log_prob + value
action, log_prob, value = policy.act(
    node_feats=batch["node_feats"],
    edge_feats=batch["edge_feats"],
    global_feats=batch["global_feats"],
    planet_mask=batch["planet_mask"],
    attack_mask=attack_mask,
    defend_mask=defend_mask,
)

n = st0.n
a_sel = action["attack_sel"][0, :n].cpu().numpy()
d_sel = action["defend_sel"][0, :n].cpu().numpy()
print(f"attack targets: {np.where(a_sel > 0.5)[0].tolist()}")
print(f"defend targets: {np.where(d_sel > 0.5)[0].tolist()}")
print(f"priority order: {action['order'][0, :n].cpu().numpy().tolist()}")
print(f"potential_weight={float(action['potential_weight'][0]):.3f}  horizon={float(action['horizon'][0]):.1f}")
print(f"log_prob={float(log_prob[0]):.3f}  value={float(value[0]):.3f}")

## 8) Scripted allocator — stratejiden launch'a

RL **hedef kümesi + öncelik + commit** verir; allocator **tek seferde** (MCF LP) gemi yönlendirir:

- **Arz kısıtı**: `Σ hedeflere giden ≤ commit_fraction[h] × ships[h]`
- **Talep**: saldırı → `garnizon + prod×ETA + margin`; savunma → `incoming_enemy − garnizon − prod×ETA`
- **Öncelik**: lexicographic (yüksek öncelikli hedef önce dolar)
- **cost_matrix**: `potential_threat` cezası + `friendly_compactness` indirimi (routing tercihi)

Yetersiz gemi varsa düşük öncelikli saldırı **düşürülür** (`attack_require_full`) — gemi harcanmaz.

In [ ]:
from feature_extraction import parse_planets
from reworked_rollout import action_to_env_launches

parsed = parse_planets(env_obs)
action_row = {
    "attack_sel": action["attack_sel"][0].cpu().numpy(),
    "defend_sel": action["defend_sel"][0].cpu().numpy(),
    "order": action["order"][0].cpu().numpy(),
    "commit_frac": action["commit_frac"][0].cpu().numpy(),
    "potential_weight": float(action["potential_weight"][0]),
    "horizon": float(action["horizon"][0]),
}

env_action, launches = action_to_env_launches(
    obs, PLAYER, action_row, parsed, av, routing_cost=True,
)
print(f"{len(launches)} launch planı → {len(env_action)} env komutu")
for L in launches[:5]:
    print(f"  src={L.src} → tgt={L.tgt}  ships={L.ships}  kind={L.kind}")
if env_action:
    print("örnek env_action [id, angle, ships]:", env_action[0])

## 9) Ödül — potential-based economic shaping

Tek ground-truth: terminal **±1** (kazan/kaybet).

Ara ödül (shaping):

```
r_t = shaping_weight · (γ · Φ(s_{t+1}) − Φ(s_t))
```

`Φ(s)` = **ekonomik potansiyel farkı** (`economic_potential`):

```
Φ = (1/norm) · [ Σ_benim (dpp_raw + w·gemi_stoğu) − Σ_düşman (…) ]
```

- `dpp_raw` = kalan ufukta üretilecek gemi
- `gemi_stoğu` = garnizon + havadaki kendi filolar
- Terminal'de `Φ := 0` → teleskopik, policy-invariant, ping-pong exploit yok

Dost filo varışı Φ'yi değiştirmez (havadan garnizona transfer); ceza/ödül **durum değişince** gelir.

In [ ]:
from reworked_rollout import economic_potential, RewardConfig

reward_cfg = RewardConfig(gamma=0.99, ship_weight=1.0)
remaining = MAX_TURNS - TURN

phi_before = economic_potential(env_obs, PLAYER, remaining, reward_cfg)
print(f"Φ(s) before step: {phi_before:.4f}")

# Bir tur oyna (her iki oyuncu da policy ile)
env_actions = [[], []]
for p in [0, 1]:
    eo = state[p]["observation"]
    o = build_observation(eo, player=p, angular_velocity=av, p_max=P_MAX, f_max=F_MAX, turn=TURN, max_turns=MAX_TURNS)
    b = pack_reworked_batch([(o, p)], device=DEVICE, p_max=P_MAX)
    am, dm = _target_masks(b["states"][0], P_MAX, p)
    act, _, _ = policy.act(
        node_feats=b["node_feats"], edge_feats=b["edge_feats"],
        global_feats=b["global_feats"], planet_mask=b["planet_mask"],
        attack_mask=torch.from_numpy(am)[None].to(DEVICE),
        defend_mask=torch.from_numpy(dm)[None].to(DEVICE),
    )
    row = {k: act[k][0].cpu().numpy() if hasattr(act[k], "cpu") else float(act[k][0])
           for k in ["attack_sel", "defend_sel", "order", "commit_frac"]}
    row["potential_weight"] = float(act["potential_weight"][0])
    row["horizon"] = float(act["horizon"][0])
    ea, _ = action_to_env_launches(o, p, row, parse_planets(eo), av)
    env_actions[p] = ea

state = env.step(env_actions)
env_obs_after = state[PLAYER]["observation"]
phi_after = economic_potential(env_obs_after, PLAYER, remaining - 1, reward_cfg)
shaped_r = reward_cfg.shaping_weight * (reward_cfg.gamma * phi_after - phi_before)
print(f"Φ(s') after step: {phi_after:.4f}")
print(f"shaped reward r_t: {shaped_r:+.6f}")

## 10) Rollout buffer + GAE + PPO

Her **tur × oyuncu** = bir `HierTransition` (padded feature'lar + aksiyon bileşenleri + log_prob + value + reward).

GAE her oyuncu için ayrı, `done=True`'da sıfırlanır → aynı buffer'da çok episode güvenle karışır.

Kredi dağıtımı:
- Shaping ödülü olayı **zamansal olarak yakına** yerleştirir
- GAE `(γλ)^gecikme` ile geriye yayar
- **Value baseline** suçu/ödülü beklenen değerin düştüğü/ yükseldiği tura lokalize eder

In [ ]:
from reworked_rollout import ReworkedBuffer, collect_episode, ppo_update, PPOConfig

buf = ReworkedBuffer()
stats = collect_episode(
    policy, buf, max_steps=15, reward_cfg=reward_cfg,
    device=DEVICE, seed=99, routing_cost=True,
)
print(f"episode length={stats.length}  launches={stats.n_launches}  winner={stats.winner}")
print(f"buffer size={len(buf)} transitions")
print(f"sample reward: {buf.data[0].reward:.6f}  done={buf.data[0].done}")

buf.compute_gae(gamma=0.99, lam=0.95)
print(f"sample advantage: {buf.data[0].advantage:.6f}  return: {buf.data[0].ret:.6f}")

opt = torch.optim.Adam(policy.parameters(), lr=3e-4)
ppo_stats = ppo_update(policy, buf, opt, PPOConfig(epochs=1, minibatch_size=32), device=DEVICE)
print(f"PPO: polL={ppo_stats.policy_loss:+.4f} valL={ppo_stats.value_loss:.4f} "
      f"H={ppo_stats.entropy:.2f} KL={ppo_stats.approx_kl:.4f} epochs={ppo_stats.epochs_run}")

## 11) Tam eğitim — `reworked_train`

Self-play PPO döngüsü (terminalden çalıştır):

```bash
cd Orbit_Wars
../orbit_wars_env/bin/python -m reworked_train \
  --iterations 200 \
  --episodes-per-iter 4 \
  --max-episode-steps 300 \
  --device cpu \
  --checkpoint-dir checkpoints_reworked \
  --log-path checkpoints_reworked/train_log.jsonl
```

Her iterasyon:
1. N episode self-play → `ReworkedBuffer`
2. `compute_gae(gamma, gae_lambda)`
3. `ppo_update` (KL early-stop, entropy anneal opsiyonel)
4. JSONL log + checkpoint (`latest.pt`, `policy_iterXXXXX.pt`)

Resume:
```bash
../orbit_wars_env/bin/python -m reworked_train --resume-from checkpoints_reworked/latest.pt
```

## Modül haritası

| Modül | Rol |
|-------|-----|
| `obs_builder.py` | Ham env → `Observation` (legacy-compatible, reworked de bunu kullanır) |
| `reworked_features.py` | Metrik fonksiyonları (dpp, threat, support, flux, …) |
| `reworked_pipeline.py` | `Observation` → feature tensörleri + allocator girdileri |
| `reworked_model.py` | GNN encoder + hiyerarşik policy (`act`, `evaluate_actions`) |
| `reworked_allocator.py` | MCF/greedy gemi yönlendirme |
| `reworked_rollout.py` | Episode toplama, ödül, buffer, GAE, PPO |
| `reworked_train.py` | Eğitim driver (CLI, checkpoint, log) |
| `docs/reworked_features_reference.md` | Metrik formülleri |

## Legacy'den farklar (özet)

| | Legacy (`rl/train.py`) | Reworked |
|---|---|---|
| Aksiyon | Otoregresif src→tgt→ships (launch başına transition) | Tur başına joint stratejik karar |
| Lojistik | Policy öğrenir | Scripted allocator (MCF) |
| Ödül | Çok kanallı zoo + arrival-credit | Tek potential-based Φ (dpp + gemi stoğu) |
| Feature | 34-D + karmaşık graph metrikleri | 15-D node + 4-D edge (temiz, lokal) |